# Phase 5 — Legal Statute Identification (LSI): Experiment 2 = **Knowledge Distillation** (Phase 1 Encoder-Decoder student ← Phase 5 Encoder+FT teacher)

**Teacher** — Experiment 1 model: Phase 1 bidirectional-GQA encoder (≈ 139 M params), fully fine-tuned (`lsi_encoder_full_ft_final.pt`).
**Student** — the **encoder stack of the Phase 1 encoder-decoder LM**, initialised from `best_model.pt`, shrunk to **< 50 M parameters**.

### How the student gets under 50 M parameters
| component | Phase 1 enc-dec | student | why |
|---|---|---|---|
| decoder (8 layers, cross-attention, ≈ 63 M) | kept | **dropped** | statute identification is a classification task on the encoded document; the decoder only exists for text generation |
| encoder (6 layers, d=768, GQA 12/4, SwiGLU 2048) | kept | **kept, all weights re-used** | ≈ 37.8 M |
| token embedding 50 000 × 768 = 38.4 M | kept | **pruned to the ≈ 14.5 k tokens that the LSI corpus really uses** (+ 260 byte/special tokens) | the embedding is the biggest block; rarely-used BPE tokens are rewritten losslessly as their sub-tokens (down to raw bytes) so no text is ever `<unk>` |
| classification head (attention pooling + linear, 0.27 M) + final RMSNorm | – | new | |

### Recipe (taken from the experiment-1 results — no new experiments here)
`views = head + cue_rest` (two 512-token windows, jointly pooled) · `attention-pooling head` · `pos-weighted BCE` (hard labels) · `enc_lr 1e-4, head_lr 1e-3, LLRD 0.92` · post-processing = `per-label shrunk(100) + top-1` (re-selected out-of-fold on dev, as in exp 1).

### Distillation signal (multi-label KD)
`L = α · BCE_posw(z_s, y)  +  (1-α) · T² · BCE(z_s / T, σ(z_t / T))  +  β · (1 − cos(W·f_s, f_t))`
* **logit distillation** — per-statute soft targets from the teacher (sigmoid with temperature, the multi-label analogue of softmax-KD),
* **representation distillation** — the attention-pooled 768-d document vector of the student (through a small training-only linear projector `W`) is pulled towards the teacher's pooled vector,
* **hard labels** — same pos-weighted BCE the teacher was trained with.

The teacher is run **once** over train/dev, and its logits + pooled features are cached on disk; the student then trains **6 epochs with early-stopping patience 2** (on dev macro-F1 at the best global threshold) without paying for the teacher in every step.

## 0. Configuration (edit this cell only)

In [ ]:
import os, sys, re, gc, json, math, time, heapq, random, subprocess, warnings
from pathlib import Path
from dataclasses import dataclass, fields, asdict
from collections import Counter, OrderedDict

import numpy as np
import pandas as pd
warnings.filterwarnings("ignore", category=UserWarning)

# ----------------------------- environment -----------------------------
ON_KAGGLE = Path("/kaggle").exists()
WORK = Path("/kaggle/working") if ON_KAGGLE else Path("./lsi_work")
WORK.mkdir(parents=True, exist_ok=True)

# ----------------------------- artifacts (set exact paths; "" = auto-discover under /kaggle/input or WORK) -----------------------------
PHASE1_ENCDEC_CKPT_OVERRIDE  = ""   # Phase 1 encoder-decoder checkpoint (best_model.pt)   <-- STUDENT init
PHASE5_TEACHER_CKPT_OVERRIDE = ""   # lsi_encoder_full_ft_final.pt from experiment 1       <-- TEACHER
PHASE1_TOKENIZER_OVERRIDE    = "/kaggle/input/datasets/vishwaspathania/tokenizer-phase5/legal_bpe_a1_50k_final.json"
TOK_CACHE_OVERRIDE           = ""   # folder "lsi_tok_cache_v1" produced by the exp-1 notebook (saves ~20-30 min). "" = auto-discover, else rebuild.

# Data: local folder holding train.json/dev.json/test.json, or download from HF (token from env HF_TOKEN / Kaggle secret / prompt).
LOCAL_DATA_DIR = ""
HF_REPO_ID = "Exploration-Lab/CS-781-Capstone"
HF_FILES = {"train": "Phase_5_LSI/Train/train.json", "dev": "Phase_5_LSI/Train/dev.json", "test": "Phase_5_LSI/Test/test.json"}

NUM_LABELS = 100
MAX_LEN = 512                 # student/teacher context per view (BOS + 510 content + EOS) -- identical to the teacher
CONTENT = MAX_LEN - 2
MAX_DOC_TOKENS = 16_384
TOK_CACHE_VERSION = "v1"

# ----------------------------- student size budget -----------------------------
PARAM_LIMIT = 50_000_000      # hard limit (training-time model INCLUDING the KD projector must stay below it)
VOCAB_KEEP_REQUEST = 14_500   # pruned vocabulary size; automatically capped so that the model fits PARAM_LIMIT

# ----------------------------- recipe found in experiment 1 (no re-tuning here) -----------------------------
STUDENT_RC = dict(views=("head", "cue_rest"), view_agg="tokens", head="attn", loss="bce_posw",
                  enc_lr=1e-4, head_lr=1e-3, llrd=0.92, wd=0.01, dropout=0.1, warmup=0.06)
EFFECTIVE_BATCH = 32          # documents per optimizer step

# ----------------------------- knowledge distillation -----------------------------
KD_ALPHA_HARD = 0.5           # weight of the hard-label BCE   (soft-label KD gets 1 - alpha)
KD_TEMP       = 2.0           # temperature of the soft targets
KD_FEAT_W     = 0.5           # weight of the pooled-representation (cosine) distillation

# ----------------------------- final training -----------------------------
FINAL_EPOCHS   = 6
FINAL_PATIENCE = 2            # early stopping on dev macro-F1 (best global threshold)
RESUME_FINAL   = False        # True -> continue an interrupted run from <tag>_last.pt
RUN_NO_KD_BASELINE = False    # True -> afterwards also train the SAME student with hard labels only (clean KD ablation for the report; +~2 h)

SMOKE_TEST = False            # True -> tiny end-to-end dry run to verify the pipeline
SEED = 42

if SMOKE_TEST: FINAL_EPOCHS = 2
RUN_DIR = WORK / ("kd_runs_smoke" if SMOKE_TEST else "kd_runs"); RUN_DIR.mkdir(parents=True, exist_ok=True)

def discover_cache():
    need = ["train_done.flag", "dev_done.flag", "test_done.flag"]
    cands = []
    if TOK_CACHE_OVERRIDE: cands.append(Path(TOK_CACHE_OVERRIDE))
    cands.append(WORK / f"lsi_tok_cache_{TOK_CACHE_VERSION}")
    if Path("/kaggle/input").exists():
        try: cands += [p.parent for p in Path("/kaggle/input").rglob("train_done.flag")]
        except Exception: pass
    for c in cands:
        if all((c / f).exists() for f in need): return c, True
    return WORK / f"lsi_tok_cache_{TOK_CACHE_VERSION}", False
CACHE_DIR, CACHE_READY = discover_cache(); CACHE_DIR.mkdir(parents=True, exist_ok=True) if not CACHE_READY else None
print("WORK:", WORK.resolve(), "| ON_KAGGLE:", ON_KAGGLE)
print("token cache:", CACHE_DIR, "(ready)" if CACHE_READY else "(will be built)")

## 1. Imports, GPU, seeds

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.checkpoint import checkpoint
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
try:
    from IPython.display import display
except Exception:
    display = print

def seed_everything(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
seed_everything(SEED)

assert torch.cuda.is_available(), "Enable a GPU runtime."
device = torch.device("cuda")
gpu = torch.cuda.get_device_properties(0)
print("GPU:", gpu.name, f"| {gpu.total_memory/1024**3:.1f} GiB | torch {torch.__version__}")

## 2. Locate data and the three artifacts (tokenizer, Phase 1 enc-dec checkpoint, Phase 5 teacher checkpoint)

In [ ]:
try:
    import ijson
except ImportError:
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ijson"]); import ijson
    except Exception:
        ijson = None

def _get_hf_token():
    tok = os.environ.get("HF_TOKEN", "").strip()
    if not tok:
        try:
            from kaggle_secrets import UserSecretsClient
            tok = UserSecretsClient().get_secret("HF_TOKEN")
        except Exception:
            tok = ""
    if not tok:
        from getpass import getpass
        tok = getpass("Hugging Face token: ")
    return tok

def locate_data():
    paths = {}
    if LOCAL_DATA_DIR:
        for split in HF_FILES:
            p = Path(LOCAL_DATA_DIR) / f"{split}.json"
            if not p.is_file(): raise FileNotFoundError(p)
            paths[split] = p
        return paths
    from huggingface_hub import hf_hub_download
    token = _get_hf_token()
    for split, fn in HF_FILES.items():
        paths[split] = Path(hf_hub_download(repo_id=HF_REPO_ID, repo_type="dataset", filename=fn, token=token))
    return paths

def iter_json_records(path):
    """Yield records from a JSON array (streamed with ijson when available) or JSONL."""
    with open(path, "rb") as f:
        first = b""
        while True:
            ch = f.read(1)
            if not ch: break
            if ch not in b" \t\r\n": first = ch; break
        f.seek(0)
        if first == b"[":
            if ijson is not None:
                yield from ijson.items(f, "item")
            else:
                yield from json.load(f)
        elif first == b"{":
            for line in f:
                line = line.strip()
                if line: yield json.loads(line)
        else:
            raise ValueError(f"Unrecognised JSON layout: {path}")

def find_artifact(filename, override=""):
    if override and Path(override).is_file(): return Path(override).resolve()
    hits = set()
    for root in [Path("/kaggle/input"), WORK, Path(".")]:
        if root.exists(): hits.update(p.resolve() for p in root.rglob(filename) if p.is_file())
    hits = sorted(hits)
    if len(hits) != 1:
        raise FileNotFoundError(f"{filename}: found {len(hits)} candidates {hits}. Set the override path in the config cell.")
    return hits[0]


need_raw = not CACHE_READY
paths = locate_data() if need_raw else {}
PHASE1_TOKENIZER_PATH   = find_artifact("legal_bpe_a1_50k_final.json", PHASE1_TOKENIZER_OVERRIDE)
PHASE1_ENCDEC_PATH      = find_artifact("best_model.pt", PHASE1_ENCDEC_CKPT_OVERRIDE)
PHASE5_TEACHER_PATH     = find_artifact("lsi_encoder_full_ft_final.pt", PHASE5_TEACHER_CKPT_OVERRIDE)
print("data paths  :", {k: str(v) for k, v in paths.items()} if paths else "(token cache present, raw files not needed)")
print("tokenizer   :", PHASE1_TOKENIZER_PATH)
print("student init:", PHASE1_ENCDEC_PATH)
print("teacher     :", PHASE5_TEACHER_PATH)

## 3. Phase 1 tokenizer (identical A-1 BPE) and the encoder building blocks (teacher architecture; the student re-uses the same blocks)

In [ ]:
SPECIAL_TOKENS = ["<pad>", "<unk>", "<s>", "</s>"]
N_SPECIAL, N_BYTES = 4, 256
BASE_VOCAB = N_SPECIAL + N_BYTES
WORD_RE = re.compile(r"\S+")

class A1BPETokenizer:
    def __init__(self, itos, merges):
        self.itos = list(itos); self.merges = [tuple(m) for m in merges]
        self.vocab_size = len(self.itos)
        self.stoi = {s: i for i, s in enumerate(self.itos)}
        self._byte_to_id = {bytes([b]): N_SPECIAL + b for b in range(256)}
        self._merge_ranks = {pair: r for r, pair in enumerate(self.merges)}

    def _encode_word(self, word):                      # unchanged from Phase 1 / Phase 3
        raw = word.encode("utf-8")
        seq = [self._byte_to_id[bytes([b])] for b in raw]
        if len(seq) < 2 or not self._merge_ranks: return seq
        n = len(seq); prev = [-1] + list(range(n - 1)); nxt = list(range(1, n)) + [-1]
        active = [True] * n; heap = []
        for i in range(n - 1):
            r = self._merge_ranks.get((seq[i], seq[i + 1]))
            if r is not None: heapq.heappush(heap, (r, i, i + 1))
        while heap:
            rank, left, right = heapq.heappop(heap)
            if not active[left] or not active[right]: continue
            if nxt[left] != right or prev[right] != left: continue
            if self._merge_ranks.get((seq[left], seq[right])) != rank: continue
            seq[left] = BASE_VOCAB + rank; active[right] = False
            rn = nxt[right]; nxt[left] = rn
            if rn != -1: prev[rn] = left
            lp = prev[left]
            if lp != -1:
                nr = self._merge_ranks.get((seq[lp], seq[left]))
                if nr is not None: heapq.heappush(heap, (nr, lp, left))
            if rn != -1:
                nr = self._merge_ranks.get((seq[left], seq[rn]))
                if nr is not None: heapq.heappush(heap, (nr, left, rn))
        out, i = [], 0
        while i != -1:
            if active[i]: out.append(seq[i])
            i = nxt[i]
        return out

def load_a1_tokenizer(path):
    with open(path, "r", encoding="utf-8") as f: payload = json.load(f)
    tok = A1BPETokenizer(payload["itos"], payload["merges"])
    assert tok.vocab_size == 50_000 and tok.itos[:4] == SPECIAL_TOKENS
    return tok

phase1_tokenizer = load_a1_tokenizer(PHASE1_TOKENIZER_PATH)
PAD_ID, UNK_ID, BOS_ID, EOS_ID = 0, 1, 2, 3
SPACE_ID = N_SPECIAL + 32          # single ASCII space byte token (id 36)
assert (phase1_tokenizer.stoi["<pad>"], phase1_tokenizer.stoi["<s>"], phase1_tokenizer.stoi["</s>"]) == (0, 2, 3)


@dataclass
class ModelConfig:
    vocab_size: int = 50_000; d_model: int = 768; n_layers: int = 16; n_heads: int = 12
    n_kv_heads: int = 4; d_ff: int = 2048; max_seq_len: int = 512
    rope_base: float = 10_000.0; rms_norm_eps: float = 1e-6; dropout: float = 0.0
D_MODEL = 768

def build_rope_cache(max_seq_len, d_head, base=10_000.0):
    inv_freq = 1.0 / (base ** (torch.arange(0, d_head // 2, dtype=torch.float32) * 2 / d_head))
    freqs = torch.outer(torch.arange(max_seq_len, dtype=torch.float32), inv_freq)
    ang = torch.cat([freqs, freqs], dim=-1)
    return ang.cos(), ang.sin()

def rotate_half(x):
    x1, x2 = torch.chunk(x, 2, dim=-1)
    return torch.cat([-x2, x1], dim=-1)

def apply_rope(x, cos, sin):
    return x * cos.to(x.device, x.dtype)[None, None] + rotate_half(x) * sin.to(x.device, x.dtype)[None, None]

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__(); self.eps = eps; self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        dt = x.dtype; xf = x.float()
        return (xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight.float()).to(dt)

class SwiGLU(nn.Module):
    def __init__(self, d, ff):
        super().__init__()
        self.gate_proj = nn.Linear(d, ff, bias=False); self.up_proj = nn.Linear(d, ff, bias=False)
        self.down_proj = nn.Linear(ff, d, bias=False)
    def forward(self, x): return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))

class BidirectionalGQAAttention(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.d_model, self.n_heads, self.n_kv_heads = c.d_model, c.n_heads, c.n_kv_heads
        self.head_dim = c.d_model // c.n_heads; self.n_rep = c.n_heads // c.n_kv_heads; self.max_seq_len = c.max_seq_len
        self.q_proj = nn.Linear(c.d_model, c.n_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(c.d_model, c.n_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(c.d_model, c.n_kv_heads * self.head_dim, bias=False)
        self.out_proj = nn.Linear(c.d_model, c.d_model, bias=False)
        cos, sin = build_rope_cache(c.max_seq_len, self.head_dim, c.rope_base)
        self.register_buffer("rope_cos", cos, persistent=False); self.register_buffer("rope_sin", sin, persistent=False)
    def _split(self, x, n):
        b, t, _ = x.shape
        return x.reshape(b, t, n, self.head_dim).transpose(1, 2)
    def forward(self, x, key_mask=None):
        b, t, _ = x.shape
        if t > self.max_seq_len: raise ValueError(f"sequence {t} > {self.max_seq_len}")
        q = apply_rope(self._split(self.q_proj(x), self.n_heads), self.rope_cos[:t], self.rope_sin[:t])
        k = apply_rope(self._split(self.k_proj(x), self.n_kv_heads), self.rope_cos[:t], self.rope_sin[:t])
        v = self._split(self.v_proj(x), self.n_kv_heads)
        k = k.repeat_interleave(self.n_rep, dim=1); v = v.repeat_interleave(self.n_rep, dim=1)
        mask = None if key_mask is None else key_mask[:, None, None, :]
        a = F.scaled_dot_product_attention(q, k, v, attn_mask=mask, dropout_p=0.0, is_causal=False)
        return self.out_proj(a.transpose(1, 2).contiguous().view(b, t, self.d_model))

class EncoderBlock(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.attn_norm = RMSNorm(c.d_model, c.rms_norm_eps); self.attn = BidirectionalGQAAttention(c)
        self.ffn_norm = RMSNorm(c.d_model, c.rms_norm_eps); self.mlp = SwiGLU(c.d_model, c.d_ff)
    def forward(self, x, key_mask=None):
        x = x + self.attn(self.attn_norm(x), key_mask)
        return x + self.mlp(self.ffn_norm(x))

class LegalEncoder(nn.Module):
    def __init__(self, c, grad_ckpt=True):
        super().__init__(); self.config = c; self.grad_ckpt = grad_ckpt
        self.token_embedding = nn.Embedding(c.vocab_size, c.d_model)
        self.layers = nn.ModuleList([EncoderBlock(c) for _ in range(c.n_layers)])
        self.final_norm = RMSNorm(c.d_model, c.rms_norm_eps)
    def forward(self, input_ids, key_mask=None):
        x = self.token_embedding(input_ids)
        for layer in self.layers:
            if self.grad_ckpt and self.training and torch.is_grad_enabled():
                x = checkpoint(layer, x, key_mask, use_reentrant=False)
            else:
                x = layer(x, key_mask)
        return self.final_norm(x)

## 4. Token cache (re-used from experiment 1 when available, otherwise built once)

In [ ]:
CUE_RE = re.compile(r"<(?:SECTION|ACT)>|^(?:[Ss]ections?|[Ss]ec\.?|[Aa]rticles?|[Rr]ules?|[Aa]ct|[Aa]cts|IPC|CrPC|Cr\.P\.C\.?|Code)\W*$")
SENT_END_CHARS = (".", ";", "?", "!", ":")
FLAG_CUE, FLAG_SENT = 1, 2

class FastDocTokenizer:
    """Word-level cached version of the Phase 1 tokenizer. Output ids are identical to the original encoder
    except runs of whitespace between words are collapsed to ONE space token and leading/trailing whitespace is dropped."""
    def __init__(self, tok, max_doc_tokens=MAX_DOC_TOKENS, tail_keep=4096, cache_limit=3_000_000):
        self.tok, self.max_doc_tokens, self.tail_keep, self.cache_limit = tok, max_doc_tokens, tail_keep, cache_limit
        self.cache = {}
    def _word(self, w):
        r = self.cache.get(w)
        if r is None:
            ids = self.tok._encode_word(w)
            fl = [0] * len(ids)
            if CUE_RE.search(w): fl[0] |= FLAG_CUE
            if w.endswith(SENT_END_CHARS): fl[-1] |= FLAG_SENT
            r = (tuple(ids), tuple(fl))
            if len(self.cache) > self.cache_limit: self.cache.clear()
            self.cache[w] = r
        return r
    def encode(self, text):
        ids, fl, cursor, first = [], [], 0, True
        for m in WORD_RE.finditer(text):
            if not first: ids.append(SPACE_ID); fl.append(0)
            w_ids, w_fl = self._word(m.group(0)); ids.extend(w_ids); fl.extend(w_fl); first = False
        n = len(ids)
        if n > self.max_doc_tokens:
            cut = self.max_doc_tokens - self.tail_keep
            ids = ids[:cut] + ids[-self.tail_keep:]; fl = fl[:cut] + fl[-self.tail_keep:]
        return np.asarray(ids, dtype=np.uint16), np.asarray(fl, dtype=np.uint8), n

_WORKER = {}
def _init_worker():
    _WORKER["t"] = FastDocTokenizer(phase1_tokenizer)
def _tok_batch(texts):
    t = _WORKER["t"]; out = []
    for s in texts:
        i, f, n = t.encode(s); out.append((i, f, n))
    return out

def build_split_cache(split):
    flag = CACHE_DIR / f"{split}_done.flag"
    if flag.exists(): return
    has_labels = split != "test"
    ids_raw, texts, labels = [], [], []
    for row in tqdm(iter_json_records(paths[split]), desc=f"read {split}"):
        ids_raw.append(row["id"]); texts.append(row["text"])
        if has_labels:
            lab = sorted(set(int(x) for x in row["labels"]))
            assert all(0 <= x < NUM_LABELS for x in lab), f"bad label in {row['id']}"
            labels.append(lab)
    n_docs = len(texts)
    batches = [texts[i:i + 128] for i in range(0, n_docs, 128)]
    results = []
    try:
        import multiprocessing as mp
        ctx = mp.get_context("fork"); nproc = min(4, os.cpu_count() or 1)
        with ctx.Pool(nproc, initializer=_init_worker) as pool:
            for r in tqdm(pool.imap(_tok_batch, batches), total=len(batches), desc=f"tokenise {split}"):
                results.extend(r)
    except Exception as e:
        print("parallel tokenisation unavailable (", type(e).__name__, ") -> sequential"); _init_worker()
        results = []
        for b in tqdm(batches, desc=f"tokenise {split} (seq)"): results.extend(_tok_batch(b))
    lengths = np.array([len(r[0]) for r in results], dtype=np.int64)
    offsets = np.zeros(n_docs + 1, dtype=np.int64); offsets[1:] = np.cumsum(lengths)
    np.save(CACHE_DIR / f"{split}_tokens.npy", np.concatenate([r[0] for r in results]))
    np.save(CACHE_DIR / f"{split}_flags.npy",  np.concatenate([r[1] for r in results]))
    np.save(CACHE_DIR / f"{split}_offsets.npy", offsets)
    np.save(CACHE_DIR / f"{split}_orig_len.npy", np.array([r[2] for r in results], dtype=np.int64))
    if has_labels:
        Y = np.zeros((n_docs, NUM_LABELS), dtype=np.uint8)
        for i, lab in enumerate(labels): Y[i, lab] = 1
        np.save(CACHE_DIR / f"{split}_labels.npy", Y)
    with open(CACHE_DIR / f"{split}_case_ids.json", "w", encoding="utf-8") as f: json.dump(ids_raw, f)
    flag.write_text("ok"); del results, texts; gc.collect()
    print(f"{split}: {n_docs:,} docs | {lengths.sum():,} tokens | mean {lengths.mean():.0f} | median {np.median(lengths):.0f}")

t0 = time.time()
for s in ["train", "dev", "test"]: build_split_cache(s)
print(f"token cache ready in {(time.time()-t0)/60:.1f} min  ->  {CACHE_DIR}")

### 4b. Load cached arrays + label statistics

In [ ]:
def load_arrays(split, has_labels=True):
    d = dict(tokens=np.load(CACHE_DIR / f"{split}_tokens.npy", mmap_mode="r"),
             flags=np.load(CACHE_DIR / f"{split}_flags.npy", mmap_mode="r"),
             offsets=np.load(CACHE_DIR / f"{split}_offsets.npy"),
             orig_len=np.load(CACHE_DIR / f"{split}_orig_len.npy"))
    d["ids"] = json.load(open(CACHE_DIR / f"{split}_case_ids.json"))
    if has_labels: d["labels"] = np.load(CACHE_DIR / f"{split}_labels.npy")
    return d

ARR = {"train": load_arrays("train"), "dev": load_arrays("dev"), "test": load_arrays("test", has_labels=False)}
for s, a in ARR.items():
    L = np.diff(a["offsets"])
    cue = np.array([int((np.asarray(a["flags"][a["offsets"][i]:a["offsets"][i+1]]) & FLAG_CUE).sum()) for i in range(0, len(L), max(1, len(L)//2000))])
    print(f"{s:5s} docs={len(L):6,d} | tokens p50={np.percentile(L,50):.0f} p90={np.percentile(L,90):.0f} p99={np.percentile(L,99):.0f}"
          f" | >510 tok: {(L>CONTENT).mean()*100:.1f}% | >1020 tok: {(L>2*CONTENT).mean()*100:.1f}% | cue words/doc (sample) mean={cue.mean():.1f}")
Ytr, Ydv = ARR["train"]["labels"], ARR["dev"]["labels"]
print("labels/doc train: mean %.2f | positives per label min/median/max = %d/%d/%d" % (Ytr.sum(1).mean(), Ytr.sum(0).min(), np.median(Ytr.sum(0)), Ytr.sum(0).max()))
PRIOR = np.clip(Ytr.mean(0), 1e-5, 1 - 1e-5)                 # label prior (train) used for bias init and class weights
POS_COUNT = Ytr.sum(0).astype(np.float64)

## 5. Document views (same `head` / `cue_rest` construction as experiment 1) + dataset with optional vocabulary remapping

In [ ]:
MAX_SEG = 96            # long "sentences" are cut into pieces of at most this many tokens
CUE_CAP = 3             # a segment's cue score saturates at this many cue words
LAST_BONUS_TOKENS = 128 # segments inside the last 128 tokens get a small bonus (operative part)

def select_cue_tokens(tok, fl, budget=CONTENT, head_keep=128):
    """Relevance-based context selection. tok/fl: uint16/uint8 arrays of the (sub)document."""
    n = len(tok)
    if n <= budget: return tok
    head = tok[:head_keep]; rest = tok[head_keep:]; rfl = fl[head_keep:]; budget_rest = budget - len(head)
    nr = len(rest)
    ends = np.flatnonzero(rfl & FLAG_SENT) + 1
    bounds = [0] + ends.tolist()
    if bounds[-1] != nr: bounds.append(nr)
    starts_l, ends_l = [], []
    for a, b in zip(bounds[:-1], bounds[1:]):
        while b - a > MAX_SEG:
            starts_l.append(a); ends_l.append(a + MAX_SEG); a += MAX_SEG
        if b > a: starts_l.append(a); ends_l.append(b)
    starts = np.asarray(starts_l); ends_a = np.asarray(ends_l)
    cs = np.concatenate([[0], np.cumsum((rfl & FLAG_CUE).astype(np.int64))])
    score = np.minimum(cs[ends_a] - cs[starts], CUE_CAP).astype(np.float64)
    score += 0.5 * (starts >= nr - LAST_BONUS_TOKENS)
    score -= np.arange(len(starts)) * 1e-6                       # stable tie-break: earlier first
    order = np.argsort(-score, kind="stable")
    chosen, used = [], 0                                    # chosen: list of (start, end) inside `rest`
    for k in order:
        s0, e0 = int(starts[k]), int(ends_a[k]); ln = e0 - s0; room = budget_rest - used
        if ln <= room:
            chosen.append((s0, e0)); used += ln
        elif room >= 16:                                    # truncate one last segment to use the remaining room
            chosen.append((s0, s0 + room)); used += room
        if budget_rest - used < 8: break
    chosen.sort()
    pieces = [rest[s0:e0] for s0, e0 in chosen]
    return np.concatenate([head] + pieces) if pieces else head

def make_view(name, tok, fl):
    n = len(tok)
    if name == "head":      return tok[:CONTENT]
    if name == "tail":      return tok[max(CONTENT, n - CONTENT):] if n > CONTENT else tok[:0]
    if name == "headtail":
        if n <= CONTENT: return tok
        h = CONTENT // 2; return np.concatenate([tok[:h], tok[n - (CONTENT - h):]])
    if name == "cue":       return select_cue_tokens(tok, fl, CONTENT, head_keep=128)
    if name == "cue_rest":
        if n <= CONTENT: return tok[:0]
        return select_cue_tokens(tok[CONTENT:], fl[CONTENT:], CONTENT, head_keep=0)
    raise ValueError(name)


class LSIDataset(Dataset):
    """Cuts the requested views from the cached full document. vmap=None -> original 50k ids (teacher);
    vmap=VocabMap -> pruned student ids (a view is cut to CONTENT tokens AFTER remapping, so a view never exceeds MAX_LEN)."""
    def __init__(self, split, indices, views, with_labels=True, vmap=None):
        self.a = ARR[split]; self.indices = np.asarray(indices, dtype=np.int64); self.views = tuple(views)
        self.with_labels = with_labels; self.need_flags = any(v.startswith("cue") for v in views); self.vmap = vmap
    def __len__(self): return len(self.indices)
    def __getitem__(self, i):
        d = int(self.indices[i]); a = self.a; o0, o1 = int(a["offsets"][d]), int(a["offsets"][d + 1])
        tok = np.asarray(a["tokens"][o0:o1]); fl = np.asarray(a["flags"][o0:o1]) if self.need_flags else None
        seqs, valid = [], []
        for v in self.views:
            c = make_view(v, tok, fl); ok = len(c) > 0
            c = self.vmap(c)[:CONTENT] if self.vmap is not None else c.astype(np.int64)
            seqs.append(np.concatenate([[BOS_ID], c, [EOS_ID]]).astype(np.int64)); valid.append(ok)
        valid[0] = True
        item = {"seqs": seqs, "valid": valid, "idx": d}
        if self.with_labels: item["y"] = a["labels"][d].astype(np.float32)
        return item

def collate_lsi(batch):
    B, V = len(batch), len(batch[0]["seqs"]); T = max(len(s) for b in batch for s in b["seqs"])
    ids = torch.full((B, V, T), PAD_ID, dtype=torch.long)
    for i, b in enumerate(batch):
        for v, s in enumerate(b["seqs"]): ids[i, v, :len(s)] = torch.from_numpy(s)
    out = {"input_ids": ids, "mask": ids.ne(PAD_ID), "valid": torch.tensor([b["valid"] for b in batch]),
           "idx": torch.tensor([b["idx"] for b in batch])}
    if "y" in batch[0]: out["y"] = torch.from_numpy(np.stack([b["y"] for b in batch]))
    return out

## 6. Evaluation + post-processing toolkit (identical to experiment 1)

In [ ]:
from scipy.ndimage import uniform_filter1d
from sklearn.metrics import average_precision_score

GRID = np.round(np.arange(0.01, 0.96, 0.01), 2)
sigmoid = lambda z: 1.0 / (1.0 + np.exp(-np.clip(z, -30, 30)))

def f1_grid(y, p, grid=GRID):
    y = y.astype(bool); npos = y.sum(0); G, C = len(grid), y.shape[1]
    tp = np.empty((G, C)); fp = np.empty((G, C))
    for gi, g in enumerate(grid):
        pr = p >= g; tp[gi] = (pr & y).sum(0); fp[gi] = pr.sum(0) - tp[gi]
    fn = npos[None, :] - tp
    return 2 * tp / np.maximum(2 * tp + fp + fn, 1)

def best_global(y, p, grid=GRID):
    f1 = f1_grid(y, p, grid); macro = f1.mean(1); i = int(np.argmax(macro))
    return float(grid[i]), float(macro[i])

def fit_per_label(y, p, t_glob, min_pos=10, shrink=100.0, grid=GRID):
    f1 = uniform_filter1d(f1_grid(y, p, grid), size=3, axis=0, mode="nearest")
    thr = grid[np.argmax(f1, axis=0)].astype(np.float64)
    sup = y.sum(0).astype(np.float64)
    thr = np.where(sup < min_pos, t_glob, thr)
    w = sup / (sup + shrink) if shrink > 0 else np.ones_like(sup)
    return w * thr + (1 - w) * t_glob

def predict_binary(p, thr, ensure_one=True):
    pred = p >= thr
    if ensure_one:
        empty = ~pred.any(1)
        if empty.any(): pred[empty, np.argmax(p[empty] / thr[None, :] if np.ndim(thr) else p[empty], axis=1)] = True
    return pred.astype(np.uint8)

def label_f1(y, pred):
    y = y.astype(bool); pr = pred.astype(bool); tp = (y & pr).sum(0); fp = (~y & pr).sum(0); fn = (y & ~pr).sum(0)
    return 2 * tp / np.maximum(2 * tp + fp + fn, 1)

def prf_summary(y, pred):
    yb, pb = y.astype(bool), pred.astype(bool)
    tp = (yb & pb).sum(0); fp = (~yb & pb).sum(0); fn = (yb & ~pb).sum(0)
    prec = tp / np.maximum(tp + fp, 1); rec = tp / np.maximum(tp + fn, 1)
    return dict(macro_f1=float(label_f1(y, pred).mean()),
                micro_f1=float(2 * tp.sum() / max(2 * tp.sum() + fp.sum() + fn.sum(), 1)),
                macro_precision=float(prec.mean()), macro_recall=float(rec.mean()),
                subset_acc=float((yb == pb).all(1).mean()), avg_pred=float(pb.sum(1).mean()),
                empty_rate=float((pb.sum(1) == 0).mean()))

# ---------------- calibration / co-occurrence re-scoring (all operate on LOGITS) ----------------
def fit_temperature(y, z):
    best, bt = 1e9, 1.0
    for T in np.arange(0.5, 4.01, 0.1):
        p = np.clip(sigmoid(z / T), 1e-7, 1 - 1e-7)
        l = -(y * np.log(p) + (1 - y) * np.log(1 - p)).mean()
        if l < best: best, bt = l, T
    def tf(z2, T=bt): return z2 / T
    tf.params = {"type": "temperature", "T": float(bt)}
    return tf

def fit_platt(y, z, lam=1.0, iters=30):
    C = y.shape[1]; a = np.ones(C); b = np.zeros(C)
    for _ in range(iters):
        p = sigmoid(z * a + b); r = p - y; w = p * (1 - p)
        ga = (r * z).sum(0) + lam * (a - 1); gb = r.sum(0) + lam * b
        haa = (w * z * z).sum(0) + lam; hab = (w * z).sum(0); hbb = w.sum(0) + lam
        det = haa * hbb - hab ** 2 + 1e-12
        a -= (hbb * ga - hab * gb) / det; b -= (-hab * ga + haa * gb) / det
        a = np.clip(a, 0.05, 5.0)
    def tf(z2, a=a, b=b): return z2 * a + b
    tf.params = {"type": "platt", "a": a.tolist(), "b": b.tolist()}
    return tf

def fit_stacker(y, z, l2=0.01, steps=100, lr=0.02, seed=0):
    """z' = a*z + z@W + b  (W has zero diagonal). Adam on mean BCE (summed over labels) + 0.5*l2*||W||^2."""
    N, C = z.shape; rng = np.random.default_rng(seed)
    a = np.ones(C); b = np.zeros(C); W = np.zeros((C, C)); eye = 1 - np.eye(C)
    params = [a, b, W]; m = [np.zeros_like(q) for q in params]; v = [np.zeros_like(q) for q in params]
    zc = np.clip(z, -12, 12)
    for t in range(1, steps + 1):
        logit = zc * a + zc @ W + b; g = (sigmoid(logit) - y) / N
        ga = (g * zc).sum(0); gb = g.sum(0); gW = (zc.T @ g) * eye + l2 * W
        for k, gr in enumerate([ga, gb, gW]):
            m[k] = 0.9 * m[k] + 0.1 * gr; v[k] = 0.999 * v[k] + 0.001 * gr ** 2
            params[k] -= lr * (m[k] / (1 - 0.9 ** t)) / (np.sqrt(v[k] / (1 - 0.999 ** t)) + 1e-8)
        W *= eye
    def tf(z2, a=a, b=b, W=W): return np.clip(z2, -12, 12) * a + np.clip(z2, -12, 12) @ W + b
    tf.params = {"type": "stacker", "a": a.tolist(), "b": b.tolist(), "W": W.tolist()}
    return tf

def stack_oof(y, z, l2, K=3):
    idx = np.random.default_rng(1).permutation(len(z)); folds = np.array_split(idx, K); out = np.zeros_like(z)
    for k in range(K):
        te = folds[k]; tr = np.concatenate([folds[j] for j in range(K) if j != k])
        out[te] = fit_stacker(y[tr], z[tr], l2)(z[te])
    return out

def make_strategy(calib=None, thr="global", shrink=100.0, min_pos=10, top1=True, l2=0.01):
    def fit(y, z):
        if calib == "temp":    tf = fit_temperature(y, z); z_thr = tf(z)
        elif calib == "platt": tf = fit_platt(y, z);       z_thr = tf(z)
        elif calib == "stack": tf = fit_stacker(y, z, l2); z_thr = stack_oof(y, z, l2)   # out-of-fold logits for thresholds
        else:
            def tf(q): return q
            tf.params = {"type": "identity"}; z_thr = z
        p_tr = sigmoid(z_thr); tg, _ = best_global(y, p_tr)
        thr_vec = np.full(y.shape[1], tg) if thr == "global" else fit_per_label(y, p_tr, tg, min_pos, shrink)
        def predict(z2): return predict_binary(sigmoid(tf(z2)), thr_vec, top1)
        predict.thr = thr_vec; predict.tf = tf; return predict
    return fit

STRATEGIES = OrderedDict([
    ("global",                         make_strategy(None,    "global", top1=False)),
    ("global + top1",                  make_strategy(None,    "global")),
    ("per-label raw + top1",           make_strategy(None,    "label", shrink=0)),
    ("per-label shrunk(100) + top1",   make_strategy(None,    "label", shrink=100)),
    ("per-label shrunk(300) + top1",   make_strategy(None,    "label", shrink=300)),
    ("temperature -> global",          make_strategy("temp",  "global")),
    ("Platt -> global",                make_strategy("platt", "global")),
    ("Platt -> per-label shrunk(100)", make_strategy("platt", "label", shrink=100)),
    ("stacker -> global",              make_strategy("stack", "global")),
    ("stacker -> per-label shrunk(100)", make_strategy("stack", "label", shrink=100)),
])

def crossfit(y, z, fit_fn, K=5, seed=0):
    rng = np.random.default_rng(seed); folds = np.array_split(rng.permutation(len(z)), K); pred = np.zeros(y.shape, np.uint8)
    for k in range(K):
        te = folds[k]; tr = np.concatenate([folds[j] for j in range(K) if j != k])
        pred[te] = fit_fn(y[tr], z[tr])(z[te])
    return pred

def ece_brier(y, p, bins=15):
    pf, yf = p.ravel(), y.ravel().astype(float); edges = np.linspace(0, 1, bins + 1); ece = 0.0
    idx = np.clip(np.digitize(pf, edges) - 1, 0, bins - 1)
    for b in range(bins):
        m = idx == b
        if m.any(): ece += m.mean() * abs(pf[m].mean() - yf[m].mean())
    return float(ece), float(((pf - yf) ** 2).mean())

def macro_ap(y, p):
    v = [average_precision_score(y[:, j], p[:, j]) for j in range(y.shape[1]) if 0 < y[:, j].sum() < len(y)]
    return float(np.mean(v)) if v else 0.0

def evaluate_logits(y, z, perlabel_cv=True):
    """Metrics used to rank experiments. Primary = macro-F1 at the best single global threshold."""
    p = sigmoid(z); thr, mf1 = best_global(y, p); pred = (p >= thr).astype(np.uint8)
    out = prf_summary(y, pred); out["best_global_thr"] = thr; out["macro_f1_global"] = mf1
    pc = np.clip(p, 1e-7, 1 - 1e-7); out["bce"] = float(-(y * np.log(pc) + (1 - y) * np.log(1 - pc)).mean())
    out["macro_ap"] = macro_ap(y, p)
    if perlabel_cv:
        pr = crossfit(y, z, STRATEGIES["per-label shrunk(100) + top1"], K=4)
        out["macro_f1_perlabel_cv"] = float(label_f1(y, pr).mean())
    return out

# ---- self-test on synthetic correlated multi-label data
_rng = np.random.default_rng(0); _N, _C = 3000, 12
_lat = _rng.normal(size=(_N, 4)); _W = _rng.normal(size=(4, _C)); _s = _lat @ _W - 1.5
_y = (_rng.random((_N, _C)) < sigmoid(_s)).astype(np.uint8); _z = _s + _rng.normal(scale=1.2, size=_s.shape)
_tmp = evaluate_logits(_y, _z); print("self-test:", {k: round(v, 3) for k, v in _tmp.items() if k in ["macro_f1_global", "macro_ap", "macro_f1_perlabel_cv"]})

In [ ]:
def strategy_table(y, z, K=5):
    rows, preds = [], {}
    for name, fn in STRATEGIES.items():
        t0 = time.time(); pr = crossfit(y, z, fn, K=K); preds[name] = pr
        rows.append(dict(strategy=name, **prf_summary(y, pr), seconds=time.time() - t0))
    return pd.DataFrame(rows).sort_values("macro_f1", ascending=False).reset_index(drop=True), preds

def write_submission(case_ids, pred_bin, path):
    out = []
    for i, cid in enumerate(case_ids):
        out.append({"id": cid, "pred": sorted({int(j) for j in np.flatnonzero(pred_bin[i])})})
    assert len(out) == len(case_ids) == len({json.dumps(c) for c in case_ids}), "one unique object per test document"
    for o in out:
        assert all(isinstance(x, int) and 0 <= x < NUM_LABELS for x in o["pred"]) and len(o["pred"]) == len(set(o["pred"]))
    json.dump(out, open(path, "w")); return out

## 7. The student: Phase 1 encoder-decoder → encoder-only classifier with a pruned vocabulary

1. Load the Phase 1 enc-dec checkpoint, keep **only** `token_embedding` + `encoder_blocks.*` (the decoder is discarded).
2. Count how often each of the 50 000 BPE ids occurs in train+dev; keep the 260 special/byte ids plus the most frequent ids so that the whole model (including the training-only KD projector) stays below `PARAM_LIMIT`.
3. A dropped id is rewritten **losslessly** as the two ids it was merged from, recursively, down to bytes (`VocabMap`) — the text is never lost and never `<unk>`.
4. Rows of the Phase 1 embedding matrix are copied for the kept ids, so the student starts from the Phase 1 weights.

In [ ]:
# ---------- 1. Phase 1 enc-dec checkpoint -> keep encoder only ----------
_ck = torch.load(PHASE1_ENCDEC_PATH, map_location="cpu", weights_only=False)
P1_CFG = dict(_ck["model_config"]); _state = {}
for k, v in _ck["model_state_dict"].items():
    for pre in ("module.", "_orig_mod."):
        if k.startswith(pre): k = k[len(pre):]
    _state[k] = v
P1_TOTAL_PARAMS = sum(v.numel() for k, v in _state.items() if k != "lm_head.weight")        # lm_head is tied to the embedding
P1_EMB = _state["token_embedding.weight"].clone()
P1_ENC_RAW = {k: v.clone() for k, v in _state.items() if k.startswith("encoder_blocks.")}
P1_DEC_PARAMS = sum(v.numel() for k, v in _state.items() if k.startswith("decoder_blocks."))
del _ck, _state; gc.collect()
print("Phase 1 enc-dec config:", {k: P1_CFG[k] for k in ["vocab_size", "d_model", "n_encoder_layers", "n_decoder_layers", "n_heads", "n_kv_heads", "d_ff"]})
print(f"Phase 1 enc-dec parameters: {P1_TOTAL_PARAMS/1e6:.1f} M  (decoder blocks alone: {P1_DEC_PARAMS/1e6:.1f} M, embedding {P1_EMB.numel()/1e6:.1f} M)")
assert P1_CFG["d_model"] == D_MODEL and P1_CFG["vocab_size"] == phase1_tokenizer.vocab_size

S_LAYERS, S_HEADS, S_KV, S_FF = P1_CFG["n_encoder_layers"], P1_CFG["n_heads"], P1_CFG["n_kv_heads"], P1_CFG["d_ff"]

# ---------- 2. token statistics on the LSI corpus ----------
cnt = np.zeros(phase1_tokenizer.vocab_size, dtype=np.int64)
for split in ("train", "dev"):
    tk = ARR[split]["tokens"]
    for s0 in range(0, len(tk), 50_000_000): cnt += np.bincount(np.asarray(tk[s0:s0 + 50_000_000]), minlength=phase1_tokenizer.vocab_size)
print(f"distinct ids used in train+dev: {(cnt > 0).sum():,} of {phase1_tokenizer.vocab_size:,} | total token occurrences {cnt.sum():,}")

# ---------- exact size budget -> maximal affordable vocabulary ----------
def non_embedding_params():
    c = ModelConfig(vocab_size=1, d_model=D_MODEL, n_layers=S_LAYERS, n_heads=S_HEADS, n_kv_heads=S_KV, d_ff=S_FF, max_seq_len=MAX_LEN)
    blocks = S_LAYERS * (2 * D_MODEL + sum(p.numel() for p in BidirectionalGQAAttention(c).parameters()) + sum(p.numel() for p in SwiGLU(D_MODEL, S_FF).parameters()))
    head_p = D_MODEL * 256 + 256 + 256 + D_MODEL * NUM_LABELS + NUM_LABELS           # AttnPoolHead (proj, v, out)
    return blocks + D_MODEL + head_p + (D_MODEL * D_MODEL + D_MODEL)                  # + final RMSNorm + KD projector (training only)
FIXED = non_embedding_params()
V_MAX = (PARAM_LIMIT - 1 - FIXED) // D_MODEL
VOCAB_KEEP = int(min(VOCAB_KEEP_REQUEST, V_MAX)); assert VOCAB_KEEP > 260
print(f"non-embedding parameters (6 enc layers + norm + head + KD projector): {FIXED/1e6:.2f} M -> max vocabulary {V_MAX:,} -> using {VOCAB_KEEP:,}")

# ---------- 3. VocabMap with lossless sub-token fallback ----------
_merges = phase1_tokenizer.merges
class VocabMap:
    def __init__(self, kept_old_ids, merges, old_vocab=50_000):
        kept = np.asarray(sorted(int(k) for k in kept_old_ids), dtype=np.int64)
        assert (kept[:BASE_VOCAB] == np.arange(BASE_VOCAB)).all(), "specials + bytes must be kept"
        self.kept = kept; self.merges = merges; self.old_vocab = old_vocab
        self.lut = np.full(old_vocab, -1, dtype=np.int32); self.lut[kept] = np.arange(len(kept), dtype=np.int32); self._exp = {}
    def _expand(self, t):
        r = self._exp.get(t)
        if r is None:
            r = [int(self.lut[t])] if self.lut[t] >= 0 else self._expand(self.merges[t - BASE_VOCAB][0]) + self._expand(self.merges[t - BASE_VOCAB][1])
            self._exp[t] = r
        return r
    def __call__(self, ids):
        out = self.lut[ids]; bad = np.flatnonzero(out < 0)
        if len(bad) == 0: return out.astype(np.int64)
        pieces, prev = [], 0
        for p in bad:
            pieces.append(out[prev:p]); pieces.append(np.asarray(self._expand(int(ids[p])), dtype=np.int32)); prev = p + 1
        pieces.append(out[prev:]); return np.concatenate(pieces).astype(np.int64)

order = np.argsort(-cnt[BASE_VOCAB:], kind="stable")[:VOCAB_KEEP - BASE_VOCAB] + BASE_VOCAB
KEPT_OLD_IDS = np.concatenate([np.arange(BASE_VOCAB), order]).astype(np.int64)
VMAP = VocabMap(KEPT_OLD_IDS, _merges)
cov = cnt[VMAP.kept].sum() / cnt.sum()
_chk = np.random.default_rng(1).choice(len(ARR["dev"]["offsets"]) - 1, 400, replace=False); n_old = n_new = 0
for d in _chk:
    a = ARR["dev"]; tk = np.asarray(a["tokens"][a["offsets"][d]:a["offsets"][d + 1]])[:2000]; n_old += len(tk); n_new += len(VMAP(tk))
print(f"kept vocabulary {len(VMAP.kept):,} ids cover {cov*100:.2f}% of all token occurrences | sequence length expansion on dev sample: x{n_new/n_old:.4f}")
# the ids of specials are unchanged by the remap
assert (VMAP.lut[:4] == np.arange(4)).all()

# ---------- 4. Phase 1 weights for the student (embedding rows sliced to the kept ids) ----------
P1_ENC_STATE = dict(P1_ENC_RAW); P1_ENC_STATE["token_embedding.weight"] = P1_EMB[torch.as_tensor(VMAP.kept)].clone()
del P1_EMB, P1_ENC_RAW; gc.collect()

In [ ]:
class StudentBlock(nn.Module):                       # parameter names identical to Phase 1 EncoderBlock -> weights load 1:1
    def __init__(self, c):
        super().__init__()
        self.norm1 = RMSNorm(c.d_model, c.rms_norm_eps); self.self_attn = BidirectionalGQAAttention(c)
        self.norm2 = RMSNorm(c.d_model, c.rms_norm_eps); self.mlp = SwiGLU(c.d_model, c.d_ff)
    def forward(self, x, key_mask=None):
        x = x + self.self_attn(self.norm1(x), key_mask)
        return x + self.mlp(self.norm2(x))

class StudentEncoder(nn.Module):
    def __init__(self, c, grad_ckpt=False):
        super().__init__(); self.config = c; self.grad_ckpt = grad_ckpt
        self.token_embedding = nn.Embedding(c.vocab_size, c.d_model)
        self.encoder_blocks = nn.ModuleList([StudentBlock(c) for _ in range(c.n_layers)])
        self.final_norm = RMSNorm(c.d_model, c.rms_norm_eps)          # new (the Phase 1 encoder had no output norm; the decoder owned it)
    def forward(self, input_ids, key_mask=None):
        x = self.token_embedding(input_ids)
        for blk in self.encoder_blocks:
            x = checkpoint(blk, x, key_mask, use_reentrant=False) if (self.grad_ckpt and self.training and torch.is_grad_enabled()) else blk(x, key_mask)
        return self.final_norm(x)

class AttnPoolHead(nn.Module):       # same head as the exp-1 winner, additionally returns the pooled document vector
    def __init__(self, C, dropout, d_att=256):
        super().__init__(); self.proj = nn.Linear(D_MODEL, d_att); self.v = nn.Linear(d_att, 1, bias=False)
        self.drop = nn.Dropout(dropout); self.out = nn.Linear(D_MODEL, C)
    def forward(self, h, ids, pm):
        cm = pm & ids.ne(BOS_ID) & ids.ne(EOS_ID)
        s = self.v(torch.tanh(self.proj(h))).squeeze(-1).float().masked_fill(~cm, -1e4)
        a = torch.softmax(s, dim=1) * cm; a = a / a.sum(1, keepdim=True).clamp_min(1e-9)
        feat = (a.unsqueeze(-1).to(h.dtype) * h).sum(1)
        return self.out(self.drop(feat)), feat

class LSIKDModel(nn.Module):
    """encoder + attention-pool head; views are encoded separately and pooled JOINTLY over all their tokens (exp-1 'tokens' aggregation)."""
    def __init__(self, encoder, dropout, prior_logit, kd_proj=False):
        super().__init__(); self.encoder = encoder; self.head = AttnPoolHead(NUM_LABELS, dropout)
        with torch.no_grad(): self.head.out.bias.copy_(torch.as_tensor(prior_logit, dtype=torch.float32))
        self.kd_proj = nn.Linear(D_MODEL, D_MODEL) if kd_proj else None
    def forward(self, input_ids, mask, valid):
        B, V, T = input_ids.shape
        h = self.encoder(input_ids.reshape(B * V, T), key_mask=mask.reshape(B * V, T)).float()
        pm = mask & valid[:, :, None]
        return self.head(h.reshape(B, V * T, -1), input_ids.reshape(B, V * T), pm.reshape(B, V * T))   # (logits, pooled feature)

PRIOR_LOGIT = np.log(PRIOR / (1 - PRIOR)).astype(np.float32)
GRAD_CKPT, MB_DOCS, NUM_WORKERS, MAX_GRAD_NORM = False, 16, 2, 1.0      # overwritten by the auto-tuner below
STUDENT_RC["views"] = tuple(STUDENT_RC["views"])

def student_config():
    return ModelConfig(vocab_size=len(VMAP.kept), d_model=D_MODEL, n_layers=S_LAYERS, n_heads=S_HEADS, n_kv_heads=S_KV, d_ff=S_FF,
                       max_seq_len=MAX_LEN, rope_base=P1_CFG.get("rope_base", 10_000.0), rms_norm_eps=P1_CFG.get("rms_norm_eps", 1e-6))

def build_student(kd_proj=True, grad_ckpt=None):
    enc = StudentEncoder(student_config(), grad_ckpt=GRAD_CKPT if grad_ckpt is None else grad_ckpt)
    res = enc.load_state_dict(P1_ENC_STATE, strict=False)
    assert res.unexpected_keys == [] and res.missing_keys == ["final_norm.weight"], res          # everything but the new output norm comes from Phase 1
    return LSIKDModel(enc, STUDENT_RC["dropout"], PRIOR_LOGIT, kd_proj=kd_proj).to(device)

def deploy_params(model): return sum(p.numel() for n, p in model.named_parameters() if not n.startswith("kd_proj."))
_m = build_student(kd_proj=True)
STUDENT_TRAIN_PARAMS = sum(p.numel() for p in _m.parameters()); STUDENT_DEPLOY_PARAMS = deploy_params(_m)
EMB_P = _m.encoder.token_embedding.weight.numel(); ENC_P = sum(p.numel() for p in _m.encoder.encoder_blocks.parameters())
print(f"student: embedding {EMB_P/1e6:.2f} M + {S_LAYERS} encoder layers {ENC_P/1e6:.2f} M + head/norm {(STUDENT_DEPLOY_PARAMS-EMB_P-ENC_P)/1e6:.2f} M "
      f"= {STUDENT_DEPLOY_PARAMS/1e6:.2f} M (deployed) | {STUDENT_TRAIN_PARAMS/1e6:.2f} M incl. training-only KD projector")
assert STUDENT_TRAIN_PARAMS < PARAM_LIMIT, "student exceeds the 50M budget"
print(f"-> {STUDENT_DEPLOY_PARAMS/P1_TOTAL_PARAMS*100:.1f}% of the Phase 1 enc-dec ({P1_TOTAL_PARAMS/1e6:.1f} M); all {S_LAYERS} encoder layers start from the Phase 1 weights")
del _m; gc.collect(); torch.cuda.empty_cache()

## 8. The teacher: load Experiment 1 (encoder + full fine-tuning) and cache its outputs

The teacher sees exactly the views it was trained on (`head + cue_rest`, original 50 k ids). Its **logits** and **pooled document vectors** are computed once for every training / dev document and stored; the student's training loop then only has to index these arrays.

*Caveat to keep in mind for the report:* the teacher was trained on these very training documents, so its train-set logits are a bit more confident than on unseen text. The sanity cell below prints that gap.

In [ ]:
tck = torch.load(PHASE5_TEACHER_PATH, map_location="cpu", weights_only=False)
T_RC = dict(tck["recipe"]); T_VIEWS = tuple(T_RC["views"])
print("teacher recipe:", T_RC, "| best epoch", tck.get("best_epoch"), "| dev macro-F1 (best epoch)", tck.get("dev_macro_f1_global") or (tck.get("dev_metrics_best_epoch") or {}).get("macro_f1_global"))
assert T_RC["head"] == "attn" and T_RC["view_agg"] == "tokens", "teacher head/aggregation differ from what this notebook implements"
if tuple(STUDENT_RC["views"]) != T_VIEWS:
    print(f"[info] student views {STUDENT_RC['views']} -> set to the teacher's views {T_VIEWS} so both models see the same evidence")
    STUDENT_RC["views"] = T_VIEWS

Ntr, Ndv, Nte = len(ARR["train"]["labels"]), len(ARR["dev"]["labels"]), len(ARR["test"]["ids"])
LIMIT = 512 if SMOKE_TEST else None
final_train_idx = np.arange(Ntr) if LIMIT is None else np.sort(np.random.default_rng(SEED).permutation(Ntr)[:LIMIT])
final_dev_idx   = np.arange(Ndv) if LIMIT is None else np.arange(min(Ndv, 256))
test_idx        = np.arange(Nte) if LIMIT is None else np.arange(min(Nte, 256))
Y_final_dev = Ydv[final_dev_idx]
print(f"train docs {len(final_train_idx):,} | dev docs {len(final_dev_idx):,} | test docs {len(test_idx):,}")

@torch.inference_mode()
def forward_pass(model, ds, batch_docs, with_feat=False):
    model.eval(); L, Fe = [], []; torch.cuda.reset_peak_memory_stats(); t0 = time.perf_counter()
    loader = DataLoader(ds, batch_size=batch_docs, shuffle=False, collate_fn=collate_lsi, num_workers=NUM_WORKERS, pin_memory=True)
    for b in tqdm(loader, desc="forward", leave=False):
        ids, m, v = (b["input_ids"].to(device, non_blocking=True), b["mask"].to(device, non_blocking=True), b["valid"].to(device, non_blocking=True))
        with torch.autocast("cuda", dtype=torch.float16): z, f = model(ids, m, v)
        L.append(z.float().cpu().numpy())
        if with_feat: Fe.append(f.float().cpu().numpy().astype(np.float16))
    dt = time.perf_counter() - t0
    return np.concatenate(L), (np.concatenate(Fe) if with_feat else None), len(ds) / dt, torch.cuda.max_memory_allocated() / 1024**3

TP = dict(train_logits=RUN_DIR / "teacher_train_logits.npy", train_feats=RUN_DIR / "teacher_train_feats.npy",
          dev_logits=RUN_DIR / "teacher_dev_logits.npy", stats=RUN_DIR / "teacher_stats.json")
if all(p.exists() for p in TP.values()) and np.load(TP["train_logits"], mmap_mode="r").shape[0] == len(final_train_idx):
    print("teacher outputs found on disk -> re-using"); TEACHER_STATS = json.load(open(TP["stats"]))
else:
    tcfg = ModelConfig(**{k: v for k, v in tck["encoder_config"].items() if k in {f.name for f in fields(ModelConfig)}})
    teacher = LSIKDModel(LegalEncoder(tcfg, grad_ckpt=False), 0.0, PRIOR_LOGIT, kd_proj=False).to(device)
    teacher.load_state_dict(tck["model_state_dict"], strict=True)
    t_params = sum(p.numel() for p in teacher.parameters())
    print(f"teacher loaded: {t_params/1e6:.1f} M parameters")
    t_dev = LSIDataset("dev", final_dev_idx, T_VIEWS); t_tr = LSIDataset("train", final_train_idx, T_VIEWS)
    z_dev, _, dev_dps, dev_peak = forward_pass(teacher, t_dev, 16)
    np.save(TP["dev_logits"], z_dev.astype(np.float32))
    print(f"teacher dev pass: {dev_dps:.1f} docs/s | peak {dev_peak:.1f} GiB")
    z_tr, f_tr, tr_dps, _ = forward_pass(teacher, t_tr, 16, with_feat=True)
    np.save(TP["train_logits"], z_tr.astype(np.float32)); np.save(TP["train_feats"], f_tr)
    TEACHER_STATS = dict(params=int(t_params), dev_docs_per_s=float(dev_dps), peak_gib=float(dev_peak), ckpt_mb=Path(PHASE5_TEACHER_PATH).stat().st_size / 1024**2)
    json.dump(TEACHER_STATS, open(TP["stats"], "w")); del teacher, t_dev, t_tr; gc.collect(); torch.cuda.empty_cache()
del tck; gc.collect()

T_DEV_LOGITS = np.load(TP["dev_logits"]); T_TRAIN_LOGITS = torch.from_numpy(np.load(TP["train_logits"])); T_TRAIN_FEATS = torch.from_numpy(np.load(TP["train_feats"]))
# NB: arrays are aligned with final_train_idx; the training loop maps dataset idx -> row
ROW_OF_TRAIN = np.full(Ntr, -1, dtype=np.int64); ROW_OF_TRAIN[final_train_idx] = np.arange(len(final_train_idx))

# ---- sanity: the teacher must reproduce its experiment-1 dev score, and how overconfident is it on its own training data?
mt = evaluate_logits(Y_final_dev, T_DEV_LOGITS)
print(f"TEACHER dev  macro-F1 {mt['macro_f1_global']:.4f} @thr {mt['best_global_thr']:.2f} | micro-F1 {mt['micro_f1']:.4f} | per-label(CV) {mt['macro_f1_perlabel_cv']:.4f}   (experiment 1 reported 0.3826 global)")
sub = np.random.default_rng(0).choice(len(final_train_idx), min(4000, len(final_train_idx)), replace=False)
_p = sigmoid(T_TRAIN_LOGITS.numpy()[sub]); _y = Ytr[final_train_idx[sub]]
print(f"TEACHER train macro-F1 {label_f1(_y, (_p >= mt['best_global_thr']).astype(np.uint8)).mean():.4f} at the same threshold  (gap to dev = how much the teacher memorised its training set)")
if not SMOKE_TEST and mt["macro_f1_global"] < 0.35: print("WARNING: teacher dev macro-F1 is far below 0.3826 -> check that the right checkpoint / token cache was loaded")

## 9. Distillation training engine

* optimiser: AdamW, **layer-wise LR decay** (embedding gets `llrd^6`, top layer the full `enc_lr`), head + KD projector at `head_lr`, linear warm-up → linear decay over **6 epochs**, fp16 AMP, grad-clip 1.0 (the same engine as the teacher's);
* loss: `α·BCE_posw + (1-α)·T²·BCE(z/T, σ(z_t/T)) + β·(1 − cos(W f_s, f_t))`; with `use_kd=False` only the first term is used (clean ablation);
* micro-batch size / gradient checkpointing are auto-tuned for this GPU.

In [ ]:
def mb_accum(): return MB_DOCS, max(1, EFFECTIVE_BATCH // MB_DOCS)

def make_optimizer(model, rc):
    n = S_LAYERS; groups = {}
    for name, p in model.named_parameters():
        if not p.requires_grad: continue
        if name.startswith("encoder."):
            if "token_embedding" in name: depth = 0
            elif ".encoder_blocks." in name: depth = int(name.split("encoder_blocks.")[1].split(".")[0]) + 1
            else: depth = n
            lr = rc["enc_lr"] * (rc["llrd"] ** (n - depth))
        else: lr = rc["head_lr"]
        groups.setdefault((lr, 0.0 if p.ndim <= 1 else rc["wd"]), []).append(p)
    return torch.optim.AdamW([{"params": ps, "lr": lr, "weight_decay": wd} for (lr, wd), ps in groups.items()], betas=(0.9, 0.999))

POS_W = torch.as_tensor(np.clip(np.sqrt((Ntr - POS_COUNT) / POS_COUNT), 1, 20), dtype=torch.float32, device=device)   # exp-1 'bce_posw'
def hard_loss(z, y): return F.binary_cross_entropy_with_logits(z, y, pos_weight=POS_W)

def build_training(model, rc, n_docs, epochs):
    mb, accum = mb_accum(); steps_per_epoch = math.ceil(math.ceil(n_docs / mb) / accum); total = steps_per_epoch * epochs
    warm = max(1, int(total * rc["warmup"])); lam = lambda s: (s + 1) / warm if s < warm else max(0.0, (total - s) / max(1, total - warm))
    opt = make_optimizer(model, rc)
    return dict(opt=opt, sched=torch.optim.lr_scheduler.LambdaLR(opt, lam), scaler=torch.amp.GradScaler("cuda"), mb=mb, accum=accum, total_steps=total)

def make_loader(ds, batch, shuffle, seed=0):
    g = torch.Generator(); g.manual_seed(seed)
    return DataLoader(ds, batch_size=batch, shuffle=shuffle, generator=g if shuffle else None, collate_fn=collate_lsi,
                      num_workers=NUM_WORKERS, pin_memory=True, drop_last=False)

def to_dev(b): return (b["input_ids"].to(device, non_blocking=True), b["mask"].to(device, non_blocking=True), b["valid"].to(device, non_blocking=True))

def train_one_epoch(model, loader, T, use_kd, tag="", log_frac=0.1):
    model.train(); opt, sched, scaler, accum = T["opt"], T["sched"], T["scaler"], T["accum"]
    opt.zero_grad(set_to_none=True); acc = dict(total=0.0, hard=0.0, soft=0.0, feat=0.0); nd, t0, n_micro = 0, time.perf_counter(), len(loader)
    next_log = max(1, int(n_micro * log_frac))
    for i, b in enumerate(loader, 1):
        ids, mask, valid = to_dev(b); y = b["y"].to(device, non_blocking=True)
        with torch.autocast("cuda", dtype=torch.float16):
            z, f = model(ids, mask, valid)
        z = z.float(); l_hard = hard_loss(z, y); l_soft = l_feat = torch.zeros((), device=device)
        if use_kd:
            rows = torch.as_tensor(ROW_OF_TRAIN[b["idx"].numpy()])
            tl = T_TRAIN_LOGITS[rows].to(device, non_blocking=True); tf = T_TRAIN_FEATS[rows].to(device, non_blocking=True).float()
            l_soft = F.binary_cross_entropy_with_logits(z / KD_TEMP, torch.sigmoid(tl / KD_TEMP)) * (KD_TEMP ** 2)
            l_feat = (1.0 - F.cosine_similarity(model.kd_proj(f.float()), tf, dim=-1)).mean()
            loss = KD_ALPHA_HARD * l_hard + (1.0 - KD_ALPHA_HARD) * l_soft + KD_FEAT_W * l_feat
        else:
            loss = l_hard
        if not torch.isfinite(loss): raise RuntimeError(f"non-finite loss at micro-batch {i} ({tag})")
        scaler.scale(loss / accum).backward(); n = ids.shape[0]; nd += n
        for k, v in (("total", loss), ("hard", l_hard), ("soft", l_soft), ("feat", l_feat)): acc[k] += float(v.detach()) * n
        if i % accum == 0 or i == n_micro:
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sched.step()
        if i >= next_log or i == n_micro:
            next_log += max(1, int(n_micro * log_frac))
            print(f"   {tag} {i}/{n_micro} | loss {acc['total']/nd:.4f} (hard {acc['hard']/nd:.4f} soft {acc['soft']/nd:.4f} feat {acc['feat']/nd:.4f}) | {nd/(time.perf_counter()-t0):.1f} docs/s", flush=True)
    return {k: v / max(nd, 1) for k, v in acc.items()}, nd / (time.perf_counter() - t0)

def predict_logits(model, ds, batch_docs):
    z, _, dps, peak = forward_pass(model, ds, batch_docs, with_feat=False); return z, dps, peak
print("engine ready")

### 9b. Auto-tune micro-batch / gradient checkpointing for this GPU

In [ ]:
def autotune_memory():
    global GRAD_CKPT, MB_DOCS
    V = len(STUDENT_RC["views"]); res = []
    for ckpt, docs in [(False, 16), (True, 16), (True, 8), (True, 4)]:
        model = opt = scaler = None
        try:
            torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
            model = build_student(kd_proj=True, grad_ckpt=ckpt); model.train(); opt = make_optimizer(model, STUDENT_RC); scaler = torch.amp.GradScaler("cuda")
            ids = torch.randint(4, len(VMAP.kept), (docs, V, MAX_LEN), device=device); ids[..., 0] = BOS_ID; ids[..., -1] = EOS_ID
            mask = torch.ones_like(ids, dtype=torch.bool); valid = torch.ones(docs, V, dtype=torch.bool, device=device)
            y = (torch.rand(docs, NUM_LABELS, device=device) < 0.03).float()
            for step in range(4):
                if step == 1: torch.cuda.synchronize(); t0 = time.perf_counter()
                with torch.autocast("cuda", dtype=torch.float16): z, f = model(ids, mask, valid)
                loss = hard_loss(z.float(), y) + (1 - F.cosine_similarity(model.kd_proj(f.float()), torch.randn_like(f.float()), dim=-1)).mean()
                opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            torch.cuda.synchronize(); dps = docs * 3 / (time.perf_counter() - t0)
            res.append((dps, ckpt, docs, torch.cuda.max_memory_allocated() / 1024**3)); print(f"grad_ckpt={ckpt!s:5} docs/micro-batch={docs:2d}: {dps:5.1f} docs/s | peak {res[-1][3]:.1f} GiB")
            if not ckpt and docs == 16: break                  # fastest config fits -> done
        except torch.cuda.OutOfMemoryError:
            print(f"grad_ckpt={ckpt!s:5} docs/micro-batch={docs:2d}: OOM")
        finally:
            del model, opt, scaler; gc.collect(); torch.cuda.empty_cache()
    if not res: raise RuntimeError("Nothing fits in GPU memory")
    dps, GRAD_CKPT, MB_DOCS, _ = max(res); print(f"-> grad_ckpt={GRAD_CKPT}, {MB_DOCS} docs ({MB_DOCS*len(STUDENT_RC['views'])} sequences) per micro-batch, ~{dps:.1f} docs/s")
    return dps
STUDENT_DPS = autotune_memory()
print(f"estimated time per epoch: ~{len(final_train_idx)/STUDENT_DPS/60:.0f} min (+ dev evaluation)  ->  {FINAL_EPOCHS} epochs ~ {FINAL_EPOCHS*len(final_train_idx)/STUDENT_DPS/3600:.1f} h worst case")

## 10. FINAL TRAINING — 6 epochs, patience 2, early stopping on dev macro-F1

`run_training` trains on **all** training documents, evaluates on the **full dev set** after every epoch, keeps the best epoch (`<tag>_best.pt`, deployable weights only — the KD projector is not saved) and writes `<tag>_last.pt` every epoch so `RESUME_FINAL = True` can continue an interrupted session.

In [ ]:
def run_training(tag, use_kd, epochs=FINAL_EPOCHS, patience=FINAL_PATIENCE):
    rc = dict(STUDENT_RC); best_pt, last_pt = RUN_DIR / f"{tag}_best.pt", RUN_DIR / f"{tag}_last.pt"
    dev_logits_path, hist_path = RUN_DIR / f"{tag}_best_dev_logits.npy", RUN_DIR / f"{tag}_history.json"
    seed_everything(SEED); model = build_student(kd_proj=use_kd); FT = build_training(model, rc, len(final_train_idx), epochs)
    train_ds = LSIDataset("train", final_train_idx, rc["views"], vmap=VMAP); dev_ds = LSIDataset("dev", final_dev_idx, rc["views"], vmap=VMAP)
    history, best_f1, best_epoch, bad, start = [], -1.0, 0, 0, 1
    if RESUME_FINAL and last_pt.exists():
        ck = torch.load(last_pt, map_location="cpu", weights_only=False)
        model.load_state_dict(ck["model"]); FT["opt"].load_state_dict(ck["opt"]); FT["sched"].load_state_dict(ck["sched"]); FT["scaler"].load_state_dict(ck["scaler"])
        history, best_f1, best_epoch, bad, start = ck["history"], ck["best_f1"], ck["best_epoch"], ck["bad"], ck["epoch"] + 1
        del ck; print(f"[{tag}] resumed after epoch {start-1}; best so far {best_f1:.4f} (epoch {best_epoch})")
    print(f"[{tag}] use_kd={use_kd} | {epochs} epochs, patience {patience} | steps/epoch {FT['total_steps']//epochs} | micro-batch {FT['mb']} docs x accum {FT['accum']}")
    t_start = time.time()
    for epoch in range(start, epochs + 1):
        ep0 = time.time()
        losses, dps = train_one_epoch(model, make_loader(train_ds, FT["mb"], True, SEED + epoch), FT, use_kd, tag=f"{tag} ep{epoch}/{epochs}")
        z_dev, _, _ = predict_logits(model, dev_ds, FT["mb"] * 2); m = evaluate_logits(Y_final_dev, z_dev)
        rec = dict(epoch=epoch, train_loss=losses["total"], train_hard=losses["hard"], train_soft=losses["soft"], train_feat=losses["feat"],
                   **{k: m[k] for k in ["macro_f1_global", "best_global_thr", "macro_ap", "macro_f1_perlabel_cv", "micro_f1", "subset_acc", "macro_precision", "macro_recall", "bce"]},
                   epoch_minutes=(time.time() - ep0) / 60, train_docs_per_s=dps); history.append(rec)
        print(f"\n=== [{tag}] epoch {epoch}: loss {losses['total']:.4f} | dev macro-F1 {m['macro_f1_global']:.4f} @thr {m['best_global_thr']:.2f} | per-label(CV) {m['macro_f1_perlabel_cv']:.4f} | "
              f"macro-AP {m['macro_ap']:.4f} | micro-F1 {m['micro_f1']:.4f} | {rec['epoch_minutes']:.1f} min ===\n")
        if m["macro_f1_global"] > best_f1 + 1e-4:
            best_f1, best_epoch, bad = m["macro_f1_global"], epoch, 0
            torch.save({"model_state_dict": {k: v.detach().cpu() for k, v in model.state_dict().items() if not k.startswith("kd_proj.")}, "epoch": epoch, "dev_macro_f1_global": best_f1}, best_pt)
            np.save(dev_logits_path, z_dev.astype(np.float32)); print(f"   new best epoch {epoch} saved -> {best_pt}")
        else:
            bad += 1; print(f"   no improvement ({bad}/{patience})")
        json.dump(history, open(hist_path, "w"), indent=1)
        torch.save({"model": model.state_dict(), "opt": FT["opt"].state_dict(), "sched": FT["sched"].state_dict(), "scaler": FT["scaler"].state_dict(),
                    "history": history, "best_f1": best_f1, "best_epoch": best_epoch, "bad": bad, "epoch": epoch}, last_pt)
        if bad >= patience: print("early stopping."); break
    wall = (time.time() - t_start) / 3600
    print(f"\n[{tag}] finished: best epoch {best_epoch}, dev macro-F1 {best_f1:.4f}, {wall:.2f} h this session")
    del model, FT, train_ds, dev_ds; gc.collect(); torch.cuda.empty_cache()
    return dict(tag=tag, use_kd=use_kd, best_f1=best_f1, best_epoch=best_epoch, history=history, best_pt=best_pt, dev_logits=dev_logits_path)

KD_RUN = run_training("student_kd", use_kd=True)
display(pd.DataFrame(KD_RUN["history"]).round(4))

### 10b. (optional) same student, hard labels only — the clean ablation for "does distillation help?"  (`RUN_NO_KD_BASELINE = True`)

In [ ]:
BASE_RUN = None
if RUN_NO_KD_BASELINE:
    BASE_RUN = run_training("student_nokd", use_kd=False); display(pd.DataFrame(BASE_RUN["history"]).round(4))
else:
    print("skipped (set RUN_NO_KD_BASELINE = True to train the hard-label-only student)")

## 11. Best epoch → post-processing selected out-of-fold on dev → test predictions → `submission.json` + final `.pt`

In [ ]:
def load_best_student(run):
    m = build_student(kd_proj=False, grad_ckpt=False)
    ck = torch.load(run["best_pt"], map_location="cpu", weights_only=False)
    res = m.load_state_dict(ck["model_state_dict"], strict=True); m.eval(); return m, ck["epoch"]

student, best_epoch = load_best_student(KD_RUN)
dev_ds = LSIDataset("dev", final_dev_idx, STUDENT_RC["views"], vmap=VMAP)
z_dev_s, dev_dps_s, dev_peak_s = predict_logits(student, dev_ds, 16)      # same batch size as the teacher pass -> comparable speed / memory
STUDENT_STATS = dict(params=int(deploy_params(student)), dev_docs_per_s=float(dev_dps_s), peak_gib=float(dev_peak_s))
print(f"student (best epoch {best_epoch}) dev inference: {dev_dps_s:.1f} docs/s | peak {dev_peak_s:.2f} GiB | teacher {TEACHER_STATS['dev_docs_per_s']:.1f} docs/s | peak {TEACHER_STATS['peak_gib']:.2f} GiB")

POST_TABLE, POST_PREDS = strategy_table(Y_final_dev, z_dev_s, K=5)
print("post-processing options, out-of-fold on the full dev set (student):"); display(POST_TABLE.round(4))
POST_NAME = POST_TABLE.iloc[0]["strategy"]
post_predictor = STRATEGIES[POST_NAME](Y_final_dev, z_dev_s)             # refit on ALL dev documents
dev_oof_pred = POST_PREDS[POST_NAME]
base_row = POST_TABLE[POST_TABLE.strategy == "global"].iloc[0]
print(f"\nchosen post-processing: {POST_NAME} | OOF dev macro-F1 {POST_TABLE.iloc[0]['macro_f1']:.4f} | micro-F1 {POST_TABLE.iloc[0]['micro_f1']:.4f} | subset acc {POST_TABLE.iloc[0]['subset_acc']:.4f}  (plain global threshold: {base_row['macro_f1']:.4f})")

test_ds = LSIDataset("test", test_idx, STUDENT_RC["views"], with_labels=False, vmap=VMAP)
z_test, test_dps, _ = predict_logits(student, test_ds, 16); np.save(RUN_DIR / "student_test_logits.npy", z_test.astype(np.float32))
print(f"test inference: {len(test_idx):,} docs in {len(test_idx)/test_dps/60:.1f} min ({test_dps:.1f} docs/s)")
test_pred = post_predictor(z_test)
test_ids = [ARR["test"]["ids"][i] for i in test_idx]
submission = write_submission(test_ids, test_pred, RUN_DIR / "submission.json")
if ON_KAGGLE: write_submission(test_ids, test_pred, WORK / "submission.json")
n_lab = np.array([len(o["pred"]) for o in submission])
print(f"submission: {len(submission):,} objects | labels/doc mean {n_lab.mean():.2f}, min {n_lab.min()}, max {n_lab.max()} | empty {int((n_lab==0).sum())} | train mean {Ytr.sum(1).mean():.2f}")
print("first 3:", submission[:3])

# ---------------- final .pt ----------------
FINAL_PT = RUN_DIR / "lsi_encdec_kd_student_final.pt"
post = dict(name=POST_NAME, thresholds=np.asarray(post_predictor.thr).tolist(), calibrator=post_predictor.tf.params, ensure_at_least_one_label=True)
best_state = torch.load(KD_RUN["best_pt"], map_location="cpu", weights_only=False)["model_state_dict"]
torch.save({"model_state_dict": best_state, "student_config": asdict(student_config()), "recipe": {**STUDENT_RC, "views": list(STUDENT_RC["views"])},
            "kd": dict(alpha_hard=KD_ALPHA_HARD, temperature=KD_TEMP, feat_weight=KD_FEAT_W, teacher="phase5 exp1 encoder full fine-tuning", teacher_ckpt=str(PHASE5_TEACHER_PATH)),
            "kept_old_ids": VMAP.kept.tolist(), "old_vocab_size": VMAP.old_vocab, "num_labels": NUM_LABELS, "best_epoch": best_epoch,
            "dev_metrics_best_epoch": KD_RUN["history"][best_epoch - 1], "postprocess": post, "dev_oof_macro_f1_after_postprocess": float(POST_TABLE.iloc[0]["macro_f1"]),
            "params_deployed": STUDENT_STATS["params"], "max_seq_len": MAX_LEN, "content_tokens_per_view": CONTENT, "tokenizer_path": str(PHASE1_TOKENIZER_PATH),
            "phase1_checkpoint": str(PHASE1_ENCDEC_PATH), "experiment_2": "phase1 enc-dec (encoder only, pruned vocab) <- knowledge distillation from exp-1 encoder"}, FINAL_PT)
STUDENT_STATS["ckpt_mb"] = FINAL_PT.stat().st_size / 1024**2
print(f"final model saved: {FINAL_PT} ({STUDENT_STATS['ckpt_mb']:.0f} MB)"); del best_state; gc.collect()

## 12. Analysis for the report (§10.5): does distillation preserve multi-label performance while shrinking the model?

In [ ]:
# ---------- teacher vs student (same dev documents, same post-processing, out-of-fold) ----------
teacher_oof = crossfit(Y_final_dev, T_DEV_LOGITS, STRATEGIES[POST_NAME], K=5)
rows = []
def row(name, z, oof, stats, views):
    g = evaluate_logits(Y_final_dev, z, perlabel_cv=False); s = prf_summary(Y_final_dev, oof)
    return {"model": name, "macro-F1 (global thr)": g["macro_f1_global"], f"macro-F1 ({POST_NAME}, OOF)": s["macro_f1"], "micro-F1": s["micro_f1"], "subset acc": s["subset_acc"],
            "macro-AP": g["macro_ap"], "macro-P": s["macro_precision"], "macro-R": s["macro_recall"], "params (M)": stats["params"] / 1e6,
            "ckpt (MB)": stats["ckpt_mb"], "inference docs/s": stats["dev_docs_per_s"], "peak GPU (GiB)": stats["peak_gib"]}
rows.append(row("TEACHER  Phase1 encoder + full FT", T_DEV_LOGITS, teacher_oof, TEACHER_STATS, T_VIEWS))
rows.append(row("STUDENT  Phase1 enc-dec encoder + KD", z_dev_s, dev_oof_pred, STUDENT_STATS, STUDENT_RC["views"]))
if BASE_RUN is not None:
    zb = np.load(BASE_RUN["dev_logits"]); pb = crossfit(Y_final_dev, zb, STRATEGIES[POST_NAME], K=5)
    rows.append(row("STUDENT  same, hard labels only (no KD)", zb, pb, STUDENT_STATS, STUDENT_RC["views"]))
cmp_df = pd.DataFrame(rows); display(cmp_df.round(4)); cmp_df.to_csv(RUN_DIR / "teacher_vs_student.csv", index=False)
t, s = cmp_df.iloc[0], cmp_df.iloc[1]; pc = f"macro-F1 ({POST_NAME}, OOF)"
print(f"\nretention: macro-F1 {s['macro-F1 (global thr)']/t['macro-F1 (global thr)']*100:.1f}% (global thr), {s[pc]/t[pc]*100:.1f}% (post-processed) | micro-F1 {s['micro-F1']/t['micro-F1']*100:.1f}%")
print(f"cost: params {t['params (M)']:.1f} M -> {s['params (M)']:.1f} M ({t['params (M)']/s['params (M)']:.1f}x smaller) | checkpoint {t['ckpt (MB)']:.0f} -> {s['ckpt (MB)']:.0f} MB | inference {t['inference docs/s']:.1f} -> {s['inference docs/s']:.1f} docs/s ({s['inference docs/s']/t['inference docs/s']:.1f}x) | peak GPU {t['peak GPU (GiB)']:.2f} -> {s['peak GPU (GiB)']:.2f} GiB")
if BASE_RUN is not None: print(f"effect of distillation (same student): macro-F1 {rows[2]['macro-F1 (global thr)']:.4f} (no KD) -> {rows[1]['macro-F1 (global thr)']:.4f} (KD)")

# ---------- training curves of the student ----------
try:
    import matplotlib.pyplot as plt
    H = pd.DataFrame(KD_RUN["history"]); fig, ax = plt.subplots(1, 2, figsize=(10, 3.3))
    ax[0].plot(H.epoch, H.train_hard, label="hard BCE_posw"); ax[0].plot(H.epoch, H.train_soft, label="soft KD (T^2 scaled)"); ax[0].plot(H.epoch, H.train_feat, label="feature cos"); ax[0].set_title("training loss components"); ax[0].legend(); ax[0].grid(alpha=.3)
    ax[1].plot(H.epoch, H.macro_f1_global, marker="o", label="student dev macro-F1"); ax[1].axhline(mt["macro_f1_global"], color="r", ls="--", label="teacher"); ax[1].set_title("dev macro-F1"); ax[1].legend(); ax[1].grid(alpha=.3)
    plt.tight_layout(); plt.show()
except Exception as e: print("plot skipped:", e)

# ---------- rare statutes: teacher vs student per support quartile ----------
sup_tr = Ytr.sum(0); q = pd.qcut(pd.Series(sup_tr).rank(method="first"), 4, labels=["rarest 25%", "rare-mid", "common-mid", "most common 25%"])
pl = pd.DataFrame({"statute": np.arange(NUM_LABELS), "train_support": sup_tr, "quartile": q, "teacher_f1": label_f1(Y_final_dev, teacher_oof), "student_f1": label_f1(Y_final_dev, dev_oof_pred)})
print("\nper-statute-frequency quartile (OOF, post-processed): does the small student lose the rare statutes first?")
display(pl.groupby("quartile", observed=True)[["teacher_f1", "student_f1"]].mean().assign(retention=lambda d: d.student_f1 / d.teacher_f1).round(3))
pl["delta"] = pl.student_f1 - pl.teacher_f1; pl.to_csv(RUN_DIR / "per_label_teacher_vs_student.csv", index=False)
print("statutes where the student is clearly worse / better than the teacher:")
display(pl.sort_values("delta").head(6).round(3).reset_index(drop=True)); display(pl.sort_values("delta").tail(6).round(3).reset_index(drop=True))

# ---------- agreement with the teacher, label co-occurrence, confusions ----------
agree = ((sigmoid(z_dev_s) >= 0.5) == (sigmoid(T_DEV_LOGITS) >= 0.5)).mean(); corr = np.corrcoef(z_dev_s.ravel(), T_DEV_LOGITS.ravel())[0, 1]
print(f"\nstudent/teacher logit correlation on dev: {corr:.3f} | binary agreement at p=0.5: {agree*100:.2f}%")
yb, pb_ = Y_final_dev.astype(bool), dev_oof_pred.astype(bool)
tp = (yb & pb_).sum(0); fp = (~yb & pb_).sum(0); fn = (yb & ~pb_).sum(0)
M = (yb & ~pb_).T.astype(np.int64) @ (~yb & pb_).astype(np.int64); np.fill_diagonal(M, 0)
top = np.dstack(np.unravel_index(np.argsort(-M.ravel())[:8], M.shape))[0]
print("most common 'missed j while spurious k predicted' pairs (student):")
for j, k in top: print(f"  missed {j:>2d} / predicted-instead {k:>2d}: {M[j,k]:4d} docs")
print(f"mean predicted labels/doc: student {pb_.sum(1).mean():.2f} | teacher {teacher_oof.sum(1).mean():.2f} | truth {yb.sum(1).mean():.2f}")
print(f"\nfiles to keep: {FINAL_PT.name} | submission.json | teacher_vs_student.csv | per_label_teacher_vs_student.csv | {KD_RUN['tag']}_history.json   (all in {RUN_DIR})")